In [1]:
"""
Network preprocessing (calculate OD)

GPU acceleration via RAPIDS (cudf, cugraph) is expected.
"""

'\nNetwork preprocessing (calculate OD)\n\nGPU acceleration via RAPIDS (cudf, cugraph) is expected.\n'

In [ ]:
# ============================================================
# 0. Python version & package installation (optional for notebooks)
# ============================================================
# Python version: 3.12.9 | packaged by conda‑forge | (main, Feb 14 2025) [GCC 13.3.0]
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
     "pandas==2.2.2", "geopandas==1.0.1", "numpy==1.26.4", "shapely==2.0.7",
     "tqdm==4.67.1", "cudf==24.10.01", "cugraph==24.10.00", "--no-deps"])

import os
import time
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm

import cudf
import cugraph
os.chdir(os.getcwd())

DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "cities"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "groupednodes"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "groupedlinks"), exist_ok=True)

In [3]:
# ============================================================
# 1. Join population / links to cities
# ------------------------------------------------------------

# Load raw spatial & population files
links = gpd.read_file('raw data/2022-TM-KA-MR-LLV2 도로망(2021년 기준)/02. 링크/ad0022_2021_KA.shp')
nodes = gpd.read_file('raw data/2022-TM-KA-MR-LLV2 도로망(2021년 기준)/01. 노드/ad0102_2021_KA.shp')
cities = gpd.read_file('raw data/bnd_sigungu_00_2023_2Q.shp')
jgg_pop = pd.read_csv('raw data/2023년기준_2022년_인구총괄(총인구).txt', sep='^', header=None)

# Convert coordinates to EPSG:5179
links = links.to_crs(epsg=5179)
nodes = nodes.to_crs(epsg=5179)
cities = cities.to_crs(epsg=5179)

# Pre‑process JGG population table
jgg_pop.columns = ['year', 'jgg_id', 'pop_type', 'pop']
jgg_pop = jgg_pop[jgg_pop['pop_type'] == 'to_in_001']
jgg_pop['SIGUNGU_CD'] = jgg_pop['jgg_id'].apply(lambda x: str(x)[:5])

# Merge the aggregated population data with the cities GeoDataFrame
city_population = jgg_pop.groupby('SIGUNGU_CD')['pop'].sum().reset_index()
cities = cities.merge(city_population, left_on='SIGUNGU_CD', right_on='SIGUNGU_CD', how='left')

# Select midsize cities
midsize_cities = cities[(cities['pop'] >= 50000) & (cities['pop'] < 1000000)]
midsize_cities = midsize_cities[~midsize_cities['SIGUNGU_NM'].str.endswith(('구', '군'))]

# Map Korean city names to English
korean_to_english = {
    "동해시": "Donghae",
    "평택시": "Pyeongtaek",
    "밀양시": "Miryang",
    "순천시": "Suncheon",
    "의왕시": "Uiwang",
    "경산시": "Gyeongsan",
    "목포시": "Mokpo",
    "의정부시": "Uijeongbu",
    "당진시": "Dangjin",
    "구미시": "Gumi",
    "여수시": "Yeosu",
    "거제시": "Geoje",
    "이천시": "Icheon",
    "광양시": "Gwangyang",
    "문경시": "Mungyeong",
    "보령시": "Boryeong",
    "군산시": "Gunsan",
    "진주시": "Jinju",
    "광주시": "Gwangju",
    "오산시": "Osan",
    "군포시": "Gunpo",
    "서귀포시": "Seogwipo",
    "제주시": "Jeju",
    "김해시": "Gimhae",
    "양주시": "Yangju",
    "김천시": "Gimcheon",
    "동두천시": "Dongducheon",
    "제천시": "Jecheon",
    "과천시": "Gwacheon",
    "익산시": "Iksan",
    "논산시": "Nonsan",
    "서산시": "Seosan",
    "김제시": "Gimje",
    "영천시": "Yeongcheon",
    "경주시": "Gyeongju",
    "사천시": "Sacheon",
    "영주시": "Yeongju",
    "하남시": "Hanam",
    "세종시": "Sejong",
    "화성시": "Hwaseong",
    "시흥시": "Siheung",
    "상주시": "Sangju",
    "통영시": "Tongyeong",
    "여주시": "Yeoju",
    "나주시": "Naju",
    "김포시": "Gimpo",
    "광명시": "Gwangmyeong",
    "파주시": "Paju",
    "춘천시": "Chuncheon",
    "부천시": "Bucheon",
    "남원시": "Namwon",
    "남양주시": "Namyangju",
    "속초시": "Sokcho",
    "포천시": "Pocheon",
    "양산시": "Yangsan",
    "충주시": "Chungju",
    "정읍시": "Jeongeup",
    "구리시": "Guri",
    "강릉시": "Gangneung",
    "안성시": "Anseong",
    "원주시": "Wonju",
    "공주시": "Gongju",
    "안동시": "Andong",
    "삼척시": "Samcheok",
    "아산시": "Asan"
}

# Add English name column & save
midsize_cities["SIGUNGU_NM_EN"] = midsize_cities["SIGUNGU_NM"].map(korean_to_english)

midsize_cities.to_file('data/midsize_cities_index.gpkg', driver='GPKG', index=False)
# (≈20 s)


In [4]:
# ============================================================
# 2. Group links & nodes by city
# ------------------------------------------------------------

# Spatial join and length calculation
links_with_cities = gpd.sjoin(links, midsize_cities[['SIGUNGU_NM', 'SIGUNGU_CD', 'geometry']], how="left", predicate='within')
nodes_with_cities = gpd.sjoin(nodes, midsize_cities[['SIGUNGU_NM', 'SIGUNGU_CD', 'geometry']], how="left", predicate='within')

links_with_cities = links_with_cities[links_with_cities['SIGUNGU_CD'].notnull()]
nodes_with_cities = nodes_with_cities[nodes_with_cities['SIGUNGU_CD'].notnull()]

links_with_cities["LENGTH"] = links_with_cities.geometry.length

# Delete nodes not in each city's graph
for city_name, group in tqdm(links_with_cities.groupby('SIGUNGU_NM'), desc='deleting nodes that are not in each cities graph'):
    G_cugraph = cugraph.Graph()
    edges_cudf = cudf.DataFrame.from_pandas(group[['UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'LINK_ID', 'SIGUNGU_NM']])
    G_cugraph.from_cudf_edgelist(edges_cudf, source='UP_FROM_NO', destination='UP_TO_NODE', edge_attr='LENGTH')
    city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name]
    city_nodes = city_nodes[city_nodes['NODE_ID'].isin(G_cugraph.nodes().to_pandas())]
    nodes_with_cities.loc[nodes_with_cities['NODE_ID'].isin(city_nodes['NODE_ID']), 'in_graph'] = True

nodes_with_cities = nodes_with_cities[nodes_with_cities['in_graph'] == True]
nodes_with_cities = nodes_with_cities.drop(columns=['in_graph'])

links_with_cities.to_file("data/links_with_cities.gpkg", driver='GPKG')
nodes_with_cities.to_file("data/nodes_with_cities.gpkg", driver='GPKG')
# (≈30 s)

deleting nodes that are not in each cities graph: 100%|██████████| 65/65 [00:06<00:00,  9.57it/s]


In [ ]:
# ============================================================
# 3. Calculate OD matrices (cuGraph SSSP)
# ------------------------------------------------------------
# 3.1 Ensure output directory exists
if not os.path.exists('data/groupedod/'):
    os.makedirs('data/groupedod/')

processing_city = 0

# Loop over each city
for city_name, group in links_with_cities.groupby('SIGUNGU_NM'):
    start_time = time.time()
    processing_city += 1

    # 3.Skip if OD already computed
    if os.path.exists(f'data/groupedod/{city_name}_od.csv'):
        print(f'{processing_city}/{len(midsize_cities)} {city_name} already processed, skipping')
        continue
    
    # Build city graph in cuGraph
    G_cugraph = cugraph.Graph()
    edges_cudf = cudf.DataFrame.from_pandas(group[['UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'LINK_ID']])
    G_cugraph.from_cudf_edgelist(edges_cudf, source='UP_FROM_NO', destination='UP_TO_NODE', edge_attr='LENGTH')
    
    city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name]
    city_nodes_list = city_nodes['NODE_ID'].to_list()

    
    # Compute all‑pairs shortest paths (SSSP per source)
    od_frames = []
    def shortest_path(src):
        return cugraph.sssp(G_cugraph, source=src, edge_attr='LENGTH')

    for src in tqdm(city_nodes_list, desc=f'Processing {city_name} {processing_city}/{len(midsize_cities)}'):
        od = shortest_path(src)
        frame = pd.DataFrame({'origin': src,
                              'destination': od['vertex'].to_numpy(),
                              'distance': od['distance'].to_numpy(),
                              'predecessor': od['predecessor'].to_numpy()})
        od_frames.append(frame)
    
    # Concatenate results & post‑process distances
    od_df = pd.concat(od_frames, ignore_index=True)
    od_df = od_df[od_df['origin'] != od_df['destination']]
    od_df['origin'] = od_df['origin'].astype(str)
    od_df['destination'] = od_df['destination'].astype(str)

    node_xy = city_nodes.set_index('NODE_ID')[['X', 'Y']]
    od_df['euclidean_distance'] = np.linalg.norm(node_xy.loc[od_df['origin']].values - node_xy.loc[od_df['destination']].values, axis=1)

    od_df['distance_ratio'] = od_df['distance'] / od_df['euclidean_distance']
    
    
    
    od_df['interval'] = (od_df['euclidean_distance'] // 100).round().astype(str)
    reached = od_df[od_df['predecessor'] != '-1']
    interval_mean = reached.groupby('interval')['distance_ratio'].mean()
    od_df = od_df.merge(interval_mean.rename('mean_ratio'), on='interval', how='left')
    od_df.loc[od_df['predecessor'] == '-1', 'distance'] = od_df['euclidean_distance'] * od_df['mean_ratio']
    od_df['distance_ratio'] = od_df['distance'] / od_df['euclidean_distance']
    od_df.drop(columns=['interval', 'mean_ratio'], inplace=True)

    # Save OD matrix
    od_df.to_csv(f'data/groupedod/{city_name}_od.csv', index=False)
    print(f'{processing_city}/{len(midsize_cities)} {city_name} processed in {time.time() - start_time:.1f} s')

1/65 강릉시 already processed, skipping
2/65 거제시 already processed, skipping
3/65 경산시 already processed, skipping
4/65 경주시 already processed, skipping
5/65 공주시 already processed, skipping
6/65 과천시 already processed, skipping
7/65 광명시 already processed, skipping
8/65 광양시 already processed, skipping
9/65 광주시 already processed, skipping
10/65 구리시 already processed, skipping
11/65 구미시 already processed, skipping
12/65 군산시 already processed, skipping
13/65 군포시 already processed, skipping
14/65 김제시 already processed, skipping
15/65 김천시 already processed, skipping
16/65 김포시 already processed, skipping
17/65 김해시 already processed, skipping
18/65 나주시 already processed, skipping
19/65 남양주시 already processed, skipping
20/65 남원시 already processed, skipping
21/65 논산시 already processed, skipping
22/65 당진시 already processed, skipping
23/65 동두천시 already processed, skipping
24/65 동해시 already processed, skipping
25/65 목포시 already processed, skipping
26/65 문경시 already processed, skipping
27/65 밀양시 already p

4. Calculate centrality metrics

In [ ]:
# ============================================================
# 4. Calculate centrality metrics
# ------------------------------------------------------------
# (Original centrality block copied verbatim; only step headers added)

# Ensure output directories exist
if not os.path.exists('data/groupednodes/'):
    os.makedirs('data/groupednodes/')
if not os.path.exists('data/groupedlinks/'):
    os.makedirs('data/groupedlinks/')

# 4.2 Helper: reconstruct route from predecessor dictionary
def find_route(origin, destination, predecessors):
    route = [destination]
    while (origin, route[-1]) in predecessors:
        route.append(predecessors[(origin, route[-1])])
    return route[::-1]

# Iterate over cities and compute node/link metrics
od_files = nodes_with_cities.groupby('SIGUNGU_NM').count().sort_values('NODE_ID', ascending=True).reset_index()['SIGUNGU_NM'].tolist()
processing_city = 0

for city_name in od_files:
    start_time = time.time()
    processing_city += 1

    node_out = f'data/groupednodes/{city_name}_node_centrality.gpkg'
    link_out = f'data/groupedlinks/{city_name}_link_betweenness.gpkg'
    #if os.path.exists(node_out) and os.path.exists(link_out):
    #    print(f'{processing_city}/{len(od_files)} {city_name} already processed, skipping')
    #    continue

    print(f'\nprocessing {city_name}: {processing_city}/{len(od_files)}')

    # 4.3.1 Load OD matrix for the city
    od_df = pd.read_csv(f'data/groupedod/{city_name}_od.csv')
    od_df['origin'] = od_df['origin'].astype(str)
    od_df['destination'] = od_df['destination'].astype(str)
    od_df['predecessor'] = od_df['predecessor'].astype(str)

    od_df_reached = od_df[od_df['predecessor'] != '-1']
    predecessors = od_df_reached.set_index(['origin', 'destination'])['predecessor'].to_dict()

    node_spi_counter = {'points': Counter()}
    nodes_counter = {'points': Counter()}
    edges_counter = {'points': Counter()}

    # Reconstruct routes in chunks to save memory
    chunk_size = 10_000_000
    num_chunks = len(od_df_reached) // chunk_size + 1
    for chunk in range(num_chunks):
        start_idx = chunk * chunk_size
        end_idx = min((chunk + 1) * chunk_size, len(od_df_reached))
        routes_cache = []
        for _, row in tqdm(od_df_reached.iloc[start_idx:end_idx].iterrows(),
                           desc=f'  Processing Chunk {chunk+1}/{num_chunks}', total=end_idx - start_idx):
            route = find_route(row['origin'], row['destination'], predecessors)
            routes_cache.append(route)
        for route in tqdm(routes_cache, desc='     node counting', total=len(routes_cache)):
            node_spi_counter['points'][route[0]] += (len(route)*(len(route)-1))/2
            for node in route:
                nodes_counter['points'][node] += 1
            for i in range(len(route)-1):
                edge = (route[i], route[i+1])
                edges_counter['points'][edge] += 1
        del routes_cache

    # Merge counters back to GeoDataFrames
    city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name].copy()
    node_route_df = pd.DataFrame(nodes_counter['points'].items(), columns=['NODE_ID', 'betweenness'])
    node_spi_df = pd.DataFrame(node_spi_counter['points'].items(), columns=['NODE_ID', 'spi'])
    city_nodes = city_nodes.merge(node_route_df, on='NODE_ID', how='left')
    city_nodes = city_nodes.merge(node_spi_df, on='NODE_ID', how='left')

    city_links = links_with_cities[links_with_cities['SIGUNGU_NM'] == city_name].copy()
    edge_route_df = pd.DataFrame(edges_counter['points'].items(), columns=['edge', 'betweenness'])
    edge_route_df[['UP_FROM_NO', 'UP_TO_NODE']] = pd.DataFrame(edge_route_df['edge'].tolist(), index=edge_route_df.index)
    edge_route_df = edge_route_df.drop(columns=['edge'])
    edge_route_df['betweenness'] *= 2  # undirected graph adjustment
    city_links = city_links.merge(edge_route_df, on=['UP_FROM_NO', 'UP_TO_NODE'], how='left')

    # Closeness & straightness
    centrality_index = od_df.groupby('origin')[['distance', 'distance_ratio']].mean().reset_index()
    centrality_index.columns = ['NODE_ID', 'closeness', 'straightness']

    nodes_per_m2 = len(city_nodes) / city_links['LENGTH'].sum()
    od_df['pairs'] = ((od_df['distance']*nodes_per_m2)*((od_df['distance']*nodes_per_m2)-1))/2
    centrality_index['expectedpairs'] = od_df.groupby('origin')['pairs'].sum().reset_index()['pairs']

    city_nodes = city_nodes.merge(centrality_index, on='NODE_ID', how='left')

    # Select final columns and save
    city_nodes = city_nodes[['NODE_ID', 'SIGUNGU_NM', 'geometry', 'betweenness', 'spi', 'expectedpairs',
                             'closeness', 'straightness']] #each closeness, straightness, and betweenness is the mean distance, detour rate, and mediation of all od pairs
    city_links = city_links[['LINK_ID', 'UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'geometry',
                             'SIGUNGU_NM', 'SIGUNGU_CD', 'betweenness']]

    city_nodes.to_file(node_out, driver='GPKG', index=False)
    city_links.to_file(link_out, driver='GPKG', index=False)

    print(f'{processing_city}/{len(od_files)} {city_name} processed in {time.time() - start_time:.1f} s')



processing 과천시: 1/65


     node counting: 100%|██████████| 232858/232858 [00:01<00:00, 164229.11it/s]


1/65 과천시 processed in 4.6 s

processing 동두천시: 2/65


     node counting: 100%|██████████| 924482/924482 [00:07<00:00, 128001.20it/s]


2/65 동두천시 processed in 28.4 s

processing 구리시: 3/65


...

63/65 서귀포시 processed in 2670.0 s

processing 제주시: 64/65


     node counting: 100%|██████████| 4389018/4389018 [03:08<00:00, 23320.40it/s] 


64/65 제주시 processed in 3822.0 s

processing 화성시: 65/65


     node counting: 100%|██████████| 9198348/9198348 [04:57<00:00, 30885.76it/s] 


65/65 화성시 processed in 4825.6 s
